# Solar-Geometry Clear-Sky Power Envelope Audit

这个 Notebook 只针对一个场站、一个检查日期，逐步验证太阳位置约束的经验晴空功率包络，不训练 TabM。

与旧版不同：

- 不再在钟表时间的96个槽位分别取90%分位数；
- 使用经纬度计算太阳高度角、方位角、晴空 GHI、日出、太阳正午和日落；
- 将过去完整日映射到统一太阳相对坐标；
- 使用低自由度的分位数样条拟合平滑晴空功率包络；
- 包络在目标日期重新映射回真实物理时间；
- 所有拟合数据严格早于检查日期。

图中文字全部使用英文，避免服务器中文字体问题。Notebook 不读取未来功率数组，也不保存转换后的数据。

## Dependencies

实验环境需要：`numpy pandas matplotlib pyarrow pvlib scikit-learn scipy`。如果缺少依赖，可在环境中安装：

```bash
pip install numpy pandas matplotlib pyarrow pvlib scikit-learn scipy
```

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pvlib
from pvlib.location import Location
from sklearn.preprocessing import SplineTransformer
from sklearn.linear_model import QuantileRegressor

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
warnings.filterwarnings("ignore", category=FutureWarning)
print("pvlib:", pvlib.__version__)

## 1. Configuration

默认检查 `雅砻江` 2024 历史站，不读取 `雅砻江解放站`。如果检查其他站，只修改 `INSPECT_RAW_STATION` 和可选的 `INSPECTION_DATE`。

In [ ]:
# ===== Paths and columns =====
DATA_DIR = Path("/path/to/private/station_parquets")
FILE_GLOB = "station=*.parquet"
STATION_INFO_PATH = Path("/home/ma-user/work/tabm_multi_stations/station_info.csv")

TIMESTAMP_COL = "timestamp_win"
STATION_COL = "station"
POWER_HISTORY_COL = "observe_power"
INFO_STATION_COL = "plantid"
INFO_CAPACITY_COL = "GCCAPACITY"
INFO_LONGITUDE_COL = "LONGITUDE"
INFO_LATITUDE_COL = "LATITUDE"

# timestamp_win is assumed to correspond to observe_power[-1].
# Set -15 if the last historical point is actually 15 minutes earlier.
HISTORY_LAST_OFFSET_MINUTES = 0
TIMESTAMP_TIMEZONE = "Asia/Shanghai"

# ===== Single-station audit =====
INSPECT_RAW_STATION = "雅砻江"
INSPECTION_DATE = "2024-12-31"  # None selects the latest complete date.
LOOKBACK_DAYS = 21
MIN_COMPLETE_DAYS = 14
MIN_VALID_SLOTS_PER_DAY = 88

# ===== Solar-envelope model =====
POWER_QUANTILE = 0.90
N_SPLINE_KNOTS = 7
SPLINE_DEGREE = 3
QUANTILE_REGULARIZATION = 1e-4
MIN_SOLAR_ELEVATION_DEG = 3.0
MIN_CLEARSKY_GHI = 20.0
MIN_ENVELOPE = 0.02

# Canonical identity and missing station-info overrides.
STATION_ALIASES = {
    "雅砻江": "雅砻江",
    "雅砻江解放站": "雅砻江",
}
STATION_OVERRIDES = {
    "雅砻江": {
        "capacity": 465.0,
        "longitude": 100.5725,
        "latitude": 29.93167,
    }
}

assert 0.5 < POWER_QUANTILE < 1.0
assert 1 <= MIN_COMPLETE_DAYS <= LOOKBACK_DAYS
assert N_SPLINE_KNOTS >= 4

## 2. Load station metadata and reconstruct the 15-minute power series

只读取 `timestamp_win`、`station`、`observe_power`，每行仅提取 `observe_power[-1]`。装机容量、经纬度来自 `station_info.csv`；雅砻江使用显式覆盖值。

In [ ]:
def canonical_station(value):
    raw = str(value).strip()
    return str(STATION_ALIASES.get(raw, raw))


def last_history_value(values):
    if values is None:
        return np.nan
    array = np.asarray(values, dtype=np.float64).reshape(-1)
    if array.size == 0 or not np.isfinite(array[-1]):
        return np.nan
    return float(array[-1])


def load_station_metadata():
    if not STATION_INFO_PATH.is_file():
        raise FileNotFoundError(f"Station info not found: {STATION_INFO_PATH}")
    info = pd.read_csv(STATION_INFO_PATH, dtype={INFO_STATION_COL: "string"})
    required = {
        INFO_STATION_COL, INFO_CAPACITY_COL, INFO_LONGITUDE_COL, INFO_LATITUDE_COL
    }
    missing = required - set(info.columns)
    if missing:
        raise KeyError(f"Missing station-info columns: {sorted(missing)}")
    info = info[list(required)].copy()
    info[INFO_STATION_COL] = info[INFO_STATION_COL].astype(str).str.strip()
    for column in [INFO_CAPACITY_COL, INFO_LONGITUDE_COL, INFO_LATITUDE_COL]:
        info[column] = pd.to_numeric(info[column], errors="coerce")
    info["station"] = info[INFO_STATION_COL].map(canonical_station)

    metadata = {}
    for station, group in info.groupby("station"):
        valid = group.dropna(
            subset=[INFO_CAPACITY_COL, INFO_LONGITUDE_COL, INFO_LATITUDE_COL]
        )
        if valid.empty:
            continue
        rows = valid[[INFO_CAPACITY_COL, INFO_LONGITUDE_COL, INFO_LATITUDE_COL]].drop_duplicates()
        if len(rows) != 1:
            raise ValueError(f"Conflicting station metadata for {station}:\n{rows}")
        row = rows.iloc[0]
        metadata[str(station)] = {
            "capacity": float(row[INFO_CAPACITY_COL]),
            "longitude": float(row[INFO_LONGITUDE_COL]),
            "latitude": float(row[INFO_LATITUDE_COL]),
        }
    for station, values in STATION_OVERRIDES.items():
        metadata[canonical_station(station)] = {key: float(value) for key, value in values.items()}
    return metadata


def locate_station_file(raw_station):
    exact = DATA_DIR / f"station={raw_station}.parquet"
    if exact.is_file():
        return exact
    matches = [p for p in sorted(DATA_DIR.glob(FILE_GLOB)) if p.stem == f"station={raw_station}"]
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected one parquet for raw station={raw_station}; found {matches}"
        )
    return matches[0]


if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Set DATA_DIR first: {DATA_DIR}")

metadata_map = load_station_metadata()
canonical_name = canonical_station(INSPECT_RAW_STATION)
if canonical_name not in metadata_map:
    raise KeyError(f"No capacity/coordinates for station {canonical_name}")
station_meta = metadata_map[canonical_name]
station_path = locate_station_file(INSPECT_RAW_STATION)

raw = pd.read_parquet(
    station_path, columns=[TIMESTAMP_COL, STATION_COL, POWER_HISTORY_COL]
)
raw["station_raw"] = raw[STATION_COL].astype(str).str.strip()
raw = raw[raw["station_raw"] == str(INSPECT_RAW_STATION).strip()].copy()
raw["timestamp_naive"] = (
    pd.to_datetime(raw[TIMESTAMP_COL], errors="coerce")
    + pd.to_timedelta(HISTORY_LAST_OFFSET_MINUTES, unit="m")
)
raw["power"] = raw[POWER_HISTORY_COL].map(last_history_value)
series = raw[["timestamp_naive", "power"]].dropna().copy()
series = series.sort_values("timestamp_naive").drop_duplicates("timestamp_naive", keep="last")
series["timestamp"] = pd.DatetimeIndex(series["timestamp_naive"]).tz_localize(
    TIMESTAMP_TIMEZONE, ambiguous="NaT", nonexistent="shift_forward"
)
series = series.dropna(subset=["timestamp"]).reset_index(drop=True)
series["power_pu_raw"] = series["power"] / station_meta["capacity"]
series["power_pu"] = series["power_pu_raw"].clip(lower=0.0)
series["date"] = series["timestamp"].dt.tz_localize(None).dt.normalize()
series["slot"] = (series["timestamp"].dt.hour * 4 + series["timestamp"].dt.minute // 15).astype(int)

print({
    "raw_station": INSPECT_RAW_STATION,
    "canonical_station": canonical_name,
    "capacity": station_meta["capacity"],
    "longitude": station_meta["longitude"],
    "latitude": station_meta["latitude"],
    "rows": len(series),
    "start": series["timestamp"].min(),
    "end": series["timestamp"].max(),
})

## 3. Select a strictly prior calibration window

检查日期本身只用于事后可视化，不参与包络拟合。拟合窗口是检查日期之前的21个自然日，并过滤缺失严重的日期。

In [ ]:
daily_counts = series.groupby("date")["power_pu"].count().sort_index()
complete_dates = daily_counts[daily_counts >= MIN_VALID_SLOTS_PER_DAY].index
if len(complete_dates) < MIN_COMPLETE_DAYS + 1:
    raise ValueError("Not enough complete days for a causal audit")

if INSPECTION_DATE is None:
    inspection_date = pd.Timestamp(complete_dates.max())
else:
    inspection_date = pd.Timestamp(INSPECTION_DATE).normalize()

calibration_start = inspection_date - pd.Timedelta(days=LOOKBACK_DAYS)
calibration_dates = complete_dates[
    (complete_dates >= calibration_start) & (complete_dates < inspection_date)
]
if len(calibration_dates) < MIN_COMPLETE_DAYS:
    raise ValueError(
        f"Only {len(calibration_dates)} complete calibration days before {inspection_date.date()}"
    )

calibration = series[series["date"].isin(calibration_dates)].copy()
inspection = series[series["date"] == inspection_date].copy()
if inspection.empty:
    warnings.warn("Inspection date has no observations; envelope can still be plotted.")

assert calibration["timestamp"].max().tz_localize(None) < inspection_date
print({
    "inspection_date": inspection_date.date(),
    "calibration_start": calibration_dates.min().date(),
    "calibration_end": calibration_dates.max().date(),
    "complete_calibration_days": len(calibration_dates),
    "calibration_rows": len(calibration),
    "inspection_rows": len(inspection),
})

## 4. Add solar geometry and theoretical clear-sky GHI

`solar_u=-1` 对应日出，`solar_u=0` 对应太阳正午，`solar_u=1` 对应日落。太阳相对坐标由经纬度和日期计算，不从功率曲线反推。

In [ ]:
location = Location(
    latitude=station_meta["latitude"],
    longitude=station_meta["longitude"],
    tz=TIMESTAMP_TIMEZONE,
    name=canonical_name,
)


def add_solar_features(frame):
    result = frame.copy().sort_values("timestamp")
    times = pd.DatetimeIndex(result["timestamp"])
    solar_position = location.get_solarposition(times)
    clear_sky = location.get_clearsky(times, model="ineichen")
    result["solar_elevation"] = solar_position["apparent_elevation"].to_numpy()
    result["solar_azimuth"] = solar_position["azimuth"].to_numpy()
    result["clear_sky_ghi"] = clear_sky["ghi"].to_numpy()

    unique_dates = pd.DatetimeIndex(sorted(result["date"].unique()))
    anchor_times = unique_dates.tz_localize(TIMESTAMP_TIMEZONE) + pd.Timedelta(hours=12)
    events = location.get_sun_rise_set_transit(anchor_times, method="spa").reset_index(drop=True)
    event_table = pd.DataFrame({"date": unique_dates})
    for column in ["sunrise", "transit", "sunset"]:
        # .array preserves the timezone-aware datetime dtype during the merge.
        event_table[column] = events[column].array
    result = result.merge(event_table, on="date", how="left")

    morning = result["timestamp"] <= result["transit"]
    morning_denominator = (result["transit"] - result["sunrise"]).dt.total_seconds()
    evening_denominator = (result["sunset"] - result["transit"]).dt.total_seconds()
    result["solar_u"] = np.where(
        morning,
        -(result["transit"] - result["timestamp"]).dt.total_seconds() / morning_denominator,
        (result["timestamp"] - result["transit"]).dt.total_seconds() / evening_denominator,
    )
    result["solar_daylight"] = (
        result["solar_elevation"] >= MIN_SOLAR_ELEVATION_DEG
    ) & result["solar_u"].between(-1.0, 1.0)
    return result


calibration_solar = add_solar_features(calibration)
inspection_solar = add_solar_features(inspection) if not inspection.empty else inspection.copy()

event_audit = (
    calibration_solar[["date", "sunrise", "transit", "sunset"]]
    .drop_duplicates("date")
    .sort_values("date")
)
display(event_audit.tail().reset_index(drop=True))

## 5. Visualize the time-shift problem before fitting

左图在钟表时间下绘制过去完整日，右图使用太阳相对坐标。只有右图明显比左图收拢，才说明太阳位置确实解释了主要相位偏移。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 5), sharey=True)
colors = plt.cm.viridis(np.linspace(0.05, 0.95, len(calibration_dates)))
for color, date in zip(colors, calibration_dates):
    day = calibration_solar[calibration_solar["date"] == date].sort_values("timestamp")
    clock_hour = day["timestamp"].dt.hour + day["timestamp"].dt.minute / 60.0
    axes[0].plot(clock_hour, day["power_pu"], color=color, alpha=0.55, lw=1.0)
    daylight = day[day["solar_daylight"]].sort_values("solar_u")
    axes[1].plot(daylight["solar_u"], daylight["power_pu"], color=color, alpha=0.55, lw=1.0)

axes[0].set_title("Calibration Days in Clock Time")
axes[0].set_xlabel("Local Clock Hour")
axes[0].set_ylabel("Power / Installed Capacity")
axes[0].set_xlim(0, 24)
axes[1].set_title("Calibration Days in Solar-Relative Time")
axes[1].set_xlabel("Solar Relative Time (-1 Sunrise, 0 Transit, 1 Sunset)")
axes[1].set_xlim(-1, 1)
fig.tight_layout()
plt.show()

## 6. Fit a solar-geometry-constrained upper-quantile envelope

模型形式为：


`envelope(t) = clear_sky_GHI(t) / 1000 × smooth_station_response(solar_u)`

晴空 GHI 强制提供正确的日出、日落和季节尺度；低自由度分位数样条只学习本站上午/下午不对称、朝向和有效功率响应。它不是逐时刻拼接，也不是预测模型。

In [ ]:
fit_data = calibration_solar[
    calibration_solar["solar_daylight"]
    & (calibration_solar["clear_sky_ghi"] >= MIN_CLEARSKY_GHI)
    & calibration_solar["power_pu"].notna()
].copy()
if len(fit_data) < 300:
    raise ValueError(f"Only {len(fit_data)} daylight points; envelope fit is not reliable")

spline = SplineTransformer(
    n_knots=N_SPLINE_KNOTS,
    degree=SPLINE_DEGREE,
    include_bias=True,
    extrapolation="constant",
)
u_fit = fit_data[["solar_u"]].to_numpy(dtype=float)
basis_fit = spline.fit_transform(u_fit)
clear_ghi_scale = fit_data["clear_sky_ghi"].to_numpy(dtype=float) / 1000.0
x_fit = basis_fit * clear_ghi_scale[:, None]
y_fit = fit_data["power_pu"].to_numpy(dtype=float)

quantile_model = QuantileRegressor(
    quantile=POWER_QUANTILE,
    alpha=QUANTILE_REGULARIZATION,
    fit_intercept=False,
    solver="highs",
)
quantile_model.fit(x_fit, y_fit)


def predict_envelope(frame):
    result = frame.copy()
    result["envelope"] = 0.0
    mask = (
        result["solar_daylight"]
        & (result["clear_sky_ghi"] >= MIN_CLEARSKY_GHI)
    )
    if mask.any():
        basis = spline.transform(result.loc[mask, ["solar_u"]].to_numpy(dtype=float))
        ghi_scale = result.loc[mask, "clear_sky_ghi"].to_numpy(dtype=float) / 1000.0
        prediction = quantile_model.predict(basis * ghi_scale[:, None])
        result.loc[mask, "envelope"] = np.clip(prediction, 0.0, None)
    result.loc[result["envelope"] < MIN_ENVELOPE, "envelope"] = 0.0
    return result


calibration_fit = predict_envelope(calibration_solar)
fit_daylight = calibration_fit[calibration_fit["envelope"] >= MIN_ENVELOPE].copy()
fit_daylight["epsi"] = fit_daylight["power_pu"] / fit_daylight["envelope"]
fit_exceedance = float((fit_daylight["power_pu"] > fit_daylight["envelope"]).mean())
print({
    "fit_points": len(fit_data),
    "quantile": POWER_QUANTILE,
    "training_exceedance_fraction": fit_exceedance,
    "training_epsi_p50": float(fit_daylight["epsi"].quantile(0.50)),
    "training_epsi_p90": float(fit_daylight["epsi"].quantile(0.90)),
    "training_epsi_p99": float(fit_daylight["epsi"].quantile(0.99)),
})

## 7. Map the envelope back to the physical inspection day

检查日的96个目标时刻由经纬度和日期生成。包络使用检查日前的数据拟合，但太阳位置与理论晴空 GHI 可以安全地由未来时间戳计算，因为它们是确定性天文量。

In [ ]:
inspection_start = inspection_date.tz_localize(TIMESTAMP_TIMEZONE)
inspection_grid = pd.DataFrame({
    "timestamp": pd.date_range(inspection_start, periods=96, freq="15min"),
})
inspection_grid["timestamp_naive"] = inspection_grid["timestamp"].dt.tz_localize(None)
inspection_grid["date"] = inspection_date
inspection_grid["slot"] = np.arange(96)
inspection_geometry = add_solar_features(inspection_grid)
inspection_envelope = predict_envelope(inspection_geometry)

if not inspection_solar.empty:
    observed_columns = inspection_solar[["timestamp", "power_pu", "power_pu_raw"]]
    inspection_envelope = inspection_envelope.merge(observed_columns, on="timestamp", how="left")
else:
    inspection_envelope["power_pu"] = np.nan
    inspection_envelope["power_pu_raw"] = np.nan

valid_eval = (inspection_envelope["envelope"] >= MIN_ENVELOPE) & inspection_envelope["power_pu"].notna()
inspection_envelope["epsi"] = np.where(
    valid_eval,
    inspection_envelope["power_pu"] / inspection_envelope["envelope"],
    np.nan,
)
inspection_envelope["reconstructed_power_pu"] = (
    inspection_envelope["epsi"] * inspection_envelope["envelope"]
)

fig, axes = plt.subplots(3, 1, figsize=(15, 11), sharex=True)
axes[0].plot(inspection_envelope["timestamp"], inspection_envelope["clear_sky_ghi"], lw=1.7)
axes[0].set_title(f"Theoretical Clear-Sky GHI: {canonical_name} — {inspection_date.date()}")
axes[0].set_ylabel("Clear-Sky GHI (W/m²)")
axes[1].plot(inspection_envelope["timestamp"], inspection_envelope["envelope"], lw=2.0, label="Solar-geometry envelope")
if inspection_envelope["power_pu"].notna().any():
    axes[1].plot(inspection_envelope["timestamp"], inspection_envelope["power_pu"], lw=1.4, label="Observed power")
axes[1].set_title("Envelope Mapped Back to Physical Time")
axes[1].set_ylabel("Power / Installed Capacity")
axes[1].legend()
axes[2].plot(inspection_envelope["timestamp"], inspection_envelope["epsi"], lw=1.5, label="ePSI")
axes[2].axhline(1.0, color="black", linestyle="--", linewidth=1.2)
axes[2].set_title("Empirical Power Clear-Sky Index")
axes[2].set_ylabel("ePSI")
axes[2].set_xlabel("Physical Time")
axes[2].set_ylim(bottom=0)
axes[2].legend()
fig.tight_layout()
plt.show()

## 8. Audit statistics and rejection conditions

逆变换精确只验证数学实现。是否接受包络还要检查日出日落边界、检查日覆盖、ePSI异常比例、训练超越率以及太阳坐标下曲线是否明显收拢。

In [ ]:
if valid_eval.any():
    evaluation = inspection_envelope.loc[valid_eval].copy()
    inverse_error = evaluation["reconstructed_power_pu"] - evaluation["power_pu"]
    stats = pd.Series({
        "inspection_daylight_points": len(evaluation),
        "envelope_peak_pu": inspection_envelope["envelope"].max(),
        "observed_peak_pu": inspection_envelope["power_pu"].max(),
        "epsi_p50": evaluation["epsi"].quantile(0.50),
        "epsi_p90": evaluation["epsi"].quantile(0.90),
        "epsi_p99": evaluation["epsi"].quantile(0.99),
        "fraction_epsi_gt_1_2": (evaluation["epsi"] > 1.2).mean(),
        "fraction_epsi_gt_1_5": (evaluation["epsi"] > 1.5).mean(),
        "inverse_rmse": np.sqrt(np.mean(inverse_error ** 2)),
        "inverse_max_abs_error": np.max(np.abs(inverse_error)),
    }, name="inspection_audit")
    display(stats.to_frame().round(6))
    assert stats["inverse_max_abs_error"] < 1e-10
else:
    print("No inspection-day observations available; visual envelope audit only.")

print("\nReject the envelope before TabM if any of these are true:")
print("- solar-relative curves do not contract compared with clock-time curves")
print("- envelope daylight boundaries disagree with theoretical sunrise/sunset")
print("- envelope has nonphysical spikes, negative sections, or unexplained plateaus")
print("- ePSI > 1.5 is frequent rather than exceptional")
print("- the 21-day window contains no credible high-power conditions")
print("- results are highly sensitive to quantile or spline-knot settings")

## Interpretation boundary

这个 Notebook 验证的是：太阳几何约束能否产生物理合理、因果、可逆的本站晴空功率基准。它尚不能证明 ePSI 会提升 TabM。

建议先依次检查：

1. 雅砻江的一个夏季日期和一个冬季日期；
2. 一个普通源站的相同季节日期；
3. 连续阴天、削顶或限电时段；
4. `POWER_QUANTILE` 在0.85、0.90、0.95附近时包络是否稳定。

只有这些单站审计通过后，再考虑多站分布比较或接入 TabM。